# Homework 7. Справочник по программированию на Go


**Задача.** По русскоязычному вопросу найти запись справочника и показать подготовленный ответ. Справочник охватывает типы данных, ошибки, конкурентность, модули и тестирование. Если подходящей записи нет, бот может отказаться от ответа.

**Файлы:** `go_knowledge.json` — авторские пары; `go_questions.json` — вопросы для настройки и проверки; `go_faq.py` — реализация поиска и консольный диалог. Для повторного запуска установите зависимости в ядре: `%pip install -r requirements.txt`. Дополнительные корпуса NLTK не нужны.

## 1.1–1.3. База знаний, предобработка и Bag of Words

Одна запись содержит идентификатор темы, основной вопрос, альтернативные формулировки и варианты ответа. Базовый индекс содержит только основные вопросы. Ответы не входят в индекс. Приводим текст к нижнему регистру, заменяем ё на е, выделяем токены, удаляем стоп-слова и применяем русский стемминг Snowball. Идентификаторы API, например `append`, `mutex` и `%w`, сохраняем. В BoW каждому термину соответствует число его вхождений.

В validation подбираем пороги; test используем только для оценки. Ни одна нормализованная формулировка проверки не копирует вопрос из базы или другого набора.

In [1]:
from collections import Counter
from html import escape
from IPython.display import HTML, display
from go_faq import (
    KNOWLEDGE, QUESTIONS, NO_ANSWER, METRICS, AUTO, SHORT_LIMIT,
    Search, tokenize, load_json, choose, evaluate,
    minimum_scores, calibrate_adaptive, adaptive_metric, learn,
)

entries = load_json(KNOWLEDGE)
data = load_json(QUESTIONS)
validation, test = data["validation"], data["test"]
answers = {entry["id"]: entry["answers"][0] for entry in entries}

def table(headers, rows):
    def cell(tag, value):
        return f"<{tag}>{escape(str(value))}</{tag}>"
    head = "".join(cell("th", x) for x in headers)
    body = "".join("<tr>" + "".join(cell("td", x) for x in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

engine_bow = Search(entries, "BoW", variants=False)
engine_tfidf = Search(entries, "TF-IDF", variants=False)
engine_plus = Search(entries, "TF-IDF", variants=True)
forms = set(engine_plus.forms)
val_forms, test_forms = [set(tuple(tokenize(q)) for q, _ in split) for split in (validation, test)]
assert not (forms & val_forms or forms & test_forms or val_forms & test_forms)
assert all(want is None or want in answers for split in (validation, test) for _, want in split)
print("Пар в базе:", len(entries), "| формулировок с парафразами:", len(engine_plus.documents))
print("Словарь BoW:", len(engine_bow.vocabulary))
for label, split in [("Validation", validation), ("Test", test)]:
    known = sum(want is not None for _, want in split)
    print(f"{label}: {known} с ответом, {len(split) - known} без ответа")
example = test[0][0]
print(example, "→", tokenize(example))
table(["ID", "Вопрос", "Ответ"], [(e["id"], e["question"], e["answers"][0]) for e in entries[:4]])

Пар в базе: 20 | формулировок с парафразами: 60
Словарь BoW: 75
Validation: 20 с ответом, 10 без ответа
Test: 20 с ответом, 10 без ответа
Как в Go запустить обработчик в отдельной горутине и продолжить работу? → ['go', 'запуст', 'обработчик', 'отдельн', 'горутин', 'продолж', 'работ']


ID,Вопрос,Ответ
goroutine,Как запустить горутину в Go?,"Поставьте `go` перед вызовом функции: `go worker()`. Вызов начнёт выполняться в отдельной горутине. Если результат нужен до завершения main, организуйте ожидание."
channel,Как создать канал и передать значение через него?,"Создайте канал: `ch := make(chan int)`. Отправка: `ch <- 42`, получение: `n := <-ch`. Для небуферизованного канала отправитель и получатель должны встретиться, обычно в разных горутинах."
close,"Как закрыть канал и узнать, что он закрыт?","Закрывает канал сторона, которая гарантирует, что отправок больше не будет. В `v, ok := <-ch` значение `ok == false` означает, что канал закрыт и оставшихся значений нет. Отправка в закрытый канал вызывает панику."
select,Как ожидать события из нескольких каналов с помощью select?,"В `select` перечислите операции над каналами: `select { case v := <-ch1: use(v); case <-ch2: stop() }`. Если ни один case не готов, select блокируется; наличие default позволяет не ждать."


## 1.4–1.7. Меры сходства и проверка

- **Жаккар:** J(A, B) = |A ∩ B| / |A ∪ B|. Учитывает наличие слов, игнорирует повторения.
- **Косинус:** cos(x, y) = (x · y) / (‖x‖ ‖y‖). Сравнивает направление векторов частот. Косинусное расстояние равно 1 − cos, поэтому его минимум соответствует максимуму сходства.
- **Пирсон:** косинус после вычитания среднего из каждого вектора. При нулевой дисперсии возвращаем 0.

Для всех методов сравниваем векторы в общем словаре. Записи без общих токенов исключаем: они не становятся случайными победителями при равенстве нулевых оценок. На базовом шаге среди оставшихся выдаём лучший результат без порога. В столбце «Ответ системы» приведён полный ответ варианта BoW + косинус; в соседних столбцах — ID ответа и его сходство для всех трёх метрик.

In [2]:
expected_val = [want for _, want in validation]
expected_test = [want for _, want in test]
engines = {"BoW": engine_bow, "TF-IDF": engine_tfidf, "TF-IDF + парафразы": engine_plus}
val_ranks = {label: [engine.rank(q) for q, _ in validation] for label, engine in engines.items()}
test_ranks = {label: [engine.rank(q) for q, _ in test] for label, engine in engines.items()}

def answer_for(entry_id):
    return answers.get(entry_id, NO_ANSWER)

def hit_label(hits):
    return f"{hits[0].id} ({hits[0].score:.3f})" if hits else "—"

rows = []
for (question, want), (_, ranked) in zip(test, test_ranks["BoW"]):
    got = choose(ranked["Косинус"], -1)
    rows.append([question, answer_for(want), answer_for(got),
                 *[hit_label(ranked[m]) for m in METRICS[:3]]])
table(["Вопрос", "Правильный ответ", "Ответ системы (косинус)", *METRICS[:3]], rows)

Вопрос,Правильный ответ,Ответ системы (косинус),Жаккар,Косинус,Пирсон
Как в Go запустить обработчик в отдельной горутине и продолжить работу?,"Поставьте `go` перед вызовом функции: `go worker()`. Вызов начнёт выполняться в отдельной горутине. Если результат нужен до завершения main, организуйте ожидание.","Поставьте `go` перед вызовом функции: `go worker()`. Вызов начнёт выполняться в отдельной горутине. Если результат нужен до завершения main, организуйте ожидание.",goroutine (1.000),goroutine (1.000),goroutine (1.000)
Как передать строку по chan между двумя задачами?,"Создайте канал: `ch := make(chan int)`. Отправка: `ch <- 42`, получение: `n := <-ch`. Для небуферизованного канала отправитель и получатель должны встретиться, обычно в разных горутинах.","Создайте канал: `ch := make(chan int)`. Отправка: `ch <- 42`, получение: `n := <-ch`. Для небуферизованного канала отправитель и получатель должны встретиться, обычно в разных горутинах.",channel (0.200),channel (0.447),channel (0.435)
"Как при получении значения определить, что отправитель уже закрыл канал?","Закрывает канал сторона, которая гарантирует, что отправок больше не будет. В `v, ok := <-ch` значение `ok == false` означает, что канал закрыт и оставшихся значений нет. Отправка в закрытый канал вызывает панику.","Закрывает канал сторона, которая гарантирует, что отправок больше не будет. В `v, ok := <-ch` значение `ok == false` означает, что канал закрыт и оставшихся значений нет. Отправка в закрытый канал вызывает панику.",close (0.500),close (0.707),close (0.696)
select по двум каналам,"В `select` перечислите операции над каналами: `select { case v := <-ch1: use(v); case <-ch2: stop() }`. Если ни один case не готов, select блокируется; наличие default позволяет не ждать.","В `select` перечислите операции над каналами: `select { case v := <-ch1: use(v); case <-ch2: stop() }`. Если ни один case не готов, select блокируется; наличие default позволяет не ждать.",select (0.333),select (0.577),select (0.561)
"Как дождаться завершения группы воркеров, если все запущены в горутинах?","Перед запуском каждой горутины вызовите `wg.Add(1)`; внутри поставьте `defer wg.Done()`. После запуска всех горутин `wg.Wait()` дождётся, пока счётчик станет нулём.","Перед запуском каждой горутины вызовите `wg.Add(1)`; внутри поставьте `defer wg.Done()`. После запуска всех горутин `wg.Wait()` дождётся, пока счётчик станет нулём.",waitgroup (0.500),waitgroup (0.707),waitgroup (0.692)
Как не дать двум горутинам одновременно поменять общий счётчик?,"Защитите чтение и запись одной и той же блокировкой: `mu.Lock(); defer mu.Unlock()`, затем работайте с общей переменной.","Защитите чтение и запись одной и той же блокировкой: `mu.Lock(); defer mu.Unlock()`, затем работайте с общей переменной.",mutex (0.333),mutex (0.516),mutex (0.491)
Как отменить запрос спустя пять секунд с помощью context?,"Создайте `ctx, cancel := context.WithTimeout(parent, time.Second)`, вызовите `defer cancel()` и передайте ctx в операцию. Она должна поддерживать контекст или проверять `ctx.Done()`.","Создайте `ctx, cancel := context.WithTimeout(parent, time.Second)`, вызовите `defer cancel()` и передайте ctx в операцию. Она должна поддерживать контекст или проверять `ctx.Done()`.",context (0.250),context (0.436),context (0.402)
Что отличает срез от обычного массива?,"У массива длина является частью типа, например `[3]int`. Слайс `[]int` описывает участок базового массива и имеет длину и ёмкость; несколько слайсов могут разделять память.","У массива длина является частью типа, например `[3]int`. Слайс `[]int` описывает участок базового массива и имеет длину и ёмкость; несколько слайсов могут разделять память.",slice (0.500),slice (0.707),slice (0.697)
append среза,"Напишите `s = append(s, value)`. append возвращает обновлённый срез; при нехватке ёмкости он может выделить новый базовый массив.","Напишите `s = append(s, value)`. append возвращает обновлённый срез; при нехватке ёмкости он может выделить новый базовый массив.",ap

Для короткого запроса из одного-двух ключевых слов Жаккар прост: лишние слова увеличивают объединение, а повтор одного термина не добавляет сходства. В длинном тексте косинус учитывает частоты и нормирует длину; с TF‑IDF он дополнительно выделяет редкие термины. Это полезная гипотеза, а не правило: длинный вопрос с множеством слов вне словаря может получить оценку по одному совпадению, а короткий запрос с редким API-токеном может отлично находиться косинусом. В конце проверим гипотезу по группам длины.

## 2.1–2.5. TF‑IDF, парафразы, варианты ответов и Спирмен

Вес термина — его частота, умноженная на IDF = ln((1 + N) / (1 + df)) + 1. Словарь и IDF строятся только по вопросам базы. Для сравнения BoW и TF‑IDF сначала держим состав базы неизменным. Затем добавляем парафразы: для каждой записи берём максимальное сходство по её формулировкам, а в Top‑3 включаем разные записи. Увеличение числа формулировок может дать как верные совпадения, так и новые ложные.

**Спирмен** — корреляция Пирсона рангов двух векторов признаков. Равные значения (включая многочисленные нули) получают средний ранг. Коэффициент учитывает относительный порядок весов, а не их абсолютные различия. Жаккар остаётся прежним при замене BoW на TF‑IDF: множество ненулевых термов то же.

In [3]:
common = sorted(engine_tfidf.df, key=lambda w: (-engine_tfidf.df[w], w))[:5]
rare = sorted(engine_tfidf.df, key=lambda w: (engine_tfidf.df[w], w))[:5]
table(["Терм", "В скольких вопросах базы", "Вес BoW при одном вхождении", "Вес TF-IDF при одном вхождении"],
      [(w, engine_tfidf.df[w], 1, f"{engine_tfidf.idf[w]:.3f}") for w in common + rare])

changes, reordered = [], []
for (question, want), (_, bow), (_, tfidf) in zip(test, test_ranks["BoW"], test_ranks["TF-IDF"]):
    a, b = choose(bow["Косинус"], -1), choose(tfidf["Косинус"], -1)
    if a != b:
        changes.append((question, want or "нет", hit_label(bow["Косинус"]), hit_label(tfidf["Косинус"])))
    left = [hit.id for hit in bow["Косинус"][:3]]
    right = [hit.id for hit in tfidf["Косинус"][:3]]
    if left != right:
        reordered.append((question, " → ".join(left), " → ".join(right)))
print("Сменился лидер косинуса при переходе от BoW к TF-IDF:", len(changes))
if changes:
    table(["Вопрос", "Эталон", "BoW", "TF-IDF"], changes)
print("Изменился порядок кандидатов в Top-3:", len(reordered))
if reordered:
    table(["Пример", "Top-3 BoW", "Top-3 TF-IDF"], reordered[:4])

Терм,В скольких вопросах базы,Вес BoW при одном вхождении,Вес TF-IDF при одном вхождении
go,8,1,1.847
через,5,1,2.253
созда,3,1,2.658
map,2,1,2.946
mod,2,1,2.946
append,1,1,3.351
context,1,1,3.351
defer,1,1,3.351
errorf,1,1,3.351
fmt,1,1,3.351


Сменился лидер косинуса при переходе от BoW к TF-IDF: 0
Изменился порядок кандидатов в Top-3: 5


Пример,Top-3 BoW,Top-3 TF-IDF
Как в Go запустить обработчик в отдельной горутине и продолжить работу?,goroutine → tests → mod_init,goroutine → tests → waitgroup
"Как проверить, есть ли заданный ключ в map, когда значение равно нулю?",map_check → map_init → channel,map_check → channel → map_init
Как получить go.mod в пустом проекте?,mod_init → mod_tidy → goroutine,mod_init → pointer → mod_tidy
Как поднять HTTP-сервер на Go?,mod_init → goroutine → interface,mod_init → goroutine → map_init


Стоп-слова, в том числе частые предлоги, исключены до взвешивания. Слова, которые остались, получают меньший **относительный** вес, если встречаются во многих вопросах. При выбранном сглаживании их абсолютный вес за одно вхождение не меньше единицы. В таблице ранжирования смена лидера сама по себе не означает улучшения: сверяем её с эталоном.

## 3.1–3.3. Пороги, автовыбор и пополнение базы

Для каждой метрики подбираем минимальный балл по validation. Перебираем границы между наблюдаемыми баллами и максимизируем среднее из Recall по известным вопросам и доли верных отказов по неизвестным. При равенстве сравниваем Accuracy, Precision, затем берём более высокий порог. Если ни одна метрика не проходит свой порог, выдаём сообщение об отсутствии ответа.

Для запроса до трёх токенов после предобработки предпочитаем Жаккара, для более длинного — косинус. Если прошедшие пороги метрики согласны по первому месту, берём предпочтительную метрику, если она прошла порог. Иначе выбираем метрику с максимальной комбинацией трёх признаков: 60 % — её сглаженная Precision на validation в данной группе длины, 25 % — нормированный запас над порогом, 15 % — отрыв от второго кандидата. Это эвристика; её качество сравниваем с фиксированными метриками.

In [4]:
limits = {label: minimum_scores(ranked, expected_val) for label, ranked in val_ranks.items()}
plus = "TF-IDF + парафразы"
reliability = calibrate_adaptive(val_ranks[plus], expected_val, limits[plus])
table(["Представление", *METRICS],
      [(label, *[f"{cut[m]:.4f}" for m in METRICS]) for label, cut in limits.items()])

updated = learn(entries, "new", "env", "Как прочитать переменную окружения?", "Используйте os.LookupEnv.")
updated = learn(updated, "variant", "env", "Как узнать значение из окружения процесса?")
updated = learn(updated, "answer", "env", "os.LookupEnv возвращает значение и признак наличия.")
demo = Search(updated)
_, demo_ranked = demo.rank("Как узнать значение из окружения процесса?")
assert demo_ranked["Косинус"][0].id == "env"
assert len(demo.entries["env"]["answers"]) == 2 and len(entries) == 20
print("Добавили в копию базы пару, парафраз и вариант ответа. После переиндексации:")
print(demo.entries["env"]["answers"])

Представление,Жаккар,Косинус,Пирсон,Спирмен
BoW,0.2679,0.3809,0.3616,0.3914
TF-IDF,0.2679,0.4297,0.4167,0.4024
TF-IDF + парафразы,0.3667,0.5201,0.5079,0.4319


Добавили в копию базы пару, парафраз и вариант ответа. После переиндексации:
['Используйте os.LookupEnv.', 'os.LookupEnv возвращает значение и признак наличия.']


Демонстрация изменяет только копию, поэтому проверочные данные не попадают в индекс эксперимента. Для реального диалога запустите в папке работы `python go_faq.py`. Команда `/all вопрос` покажет все варианты записей, прошедших порог; `/random вопрос` выберет один случайный ответ из того же множества. Без команды бот возвращает основной ответ лучшей записи. `--metric Косинус` фиксирует метрику; `--threshold 0.4` позволяет задать порог вручную. Команды обучения:

- `/teach id | вопрос | ответ` — новая пара;
- `/variant id | вопрос` — ещё одна формулировка;
- `/answer id | ответ` — альтернативный ответ.

После обучения индекс и пороги пересчитываются. Добавления сохраняются в `go_knowledge.user.json`, исходный справочник остаётся прежним. При пополнении базы имеет смысл обновлять и validation: бывший вопрос «без ответа» может стать известным.

## 3.4–3.5. Автоматическая оценка

**Accuracy** — доля полностью корректных исходов, включая отказ для неизвестного вопроса. **Top‑3 accuracy** — доля известных вопросов, у которых эталонная запись входит в первые три до отсечения порогом. **Precision** — доля корректных среди выданных ответов (ошибочная выдача считается FP). **Recall** — доля найденных правильных ответов среди известных вопросов (пропуск или неверный ответ считается FN). Ответы сверяем по ID записи: альтернативные ответы одной записи считаются эквивалентными. Для контроля отказов также считаем их долю среди вопросов без ответа.

In [5]:
def run(label, metric, ranked_rows, expected, use_threshold=True):
    predictions, rankings, used = [], [], []
    for length, ranked in ranked_rows:
        selected = adaptive_metric(length, ranked, limits[label], reliability) if metric == AUTO else metric
        cutoff = limits[label][selected] if use_threshold else -1
        rankings.append(ranked[selected])
        predictions.append(choose(ranked[selected], cutoff))
        used.append(selected)
    return evaluate(expected, predictions, rankings), predictions, rankings, used

configs = [("BoW", m, False) for m in METRICS[:3]]
configs += [(label, m, True) for label in engines for m in METRICS
            if label != "BoW" or m != "Спирмен"]
configs.append((plus, AUTO, True))
stats, stats_val = {}, {}
for label, metric, cutoff in configs:
    key = (label, metric, cutoff)
    stats[key] = run(label, metric, test_ranks[label], expected_test, cutoff)
    stats_val[key] = run(label, metric, val_ranks[label], expected_val, cutoff)[0]

columns = ["Accuracy", "Top-3 accuracy", "Precision", "Recall", "Отказы вне базы"]
table(["Представление", "Метрика", "Порог", *columns],
      [(label, metric, "да" if cutoff else "нет", *[f"{stats[label, metric, cutoff][0][c]:.3f}" for c in columns])
       for label, metric, cutoff in configs])

print("Метрики, выбранные автовыбором на test:", dict(Counter(stats[plus, AUTO, True][3])))
length_rows = []
for short in (True, False):
    idx = [i for i, (length, _) in enumerate(test_ranks[plus])
           if (length <= SHORT_LIMIT) == short and expected_test[i] is not None]
    for metric in ("Жаккар", "Косинус", AUTO):
        pred = stats[plus, metric, True][1]
        length_rows.append(("до 3 токенов" if short else "больше 3", metric,
                            sum(pred[i] == expected_test[i] for i in idx), len(idx)))
table(["Известные вопросы", "Метрика", "Отвечено верно", "Всего"], length_rows)

Представление,Метрика,Порог,Accuracy,Top-3 accuracy,Precision,Recall,Отказы вне базы
BoW,Жаккар,нет,0.700,1.000,0.654,0.850,0.400
BoW,Косинус,нет,0.700,1.000,0.654,0.850,0.400
BoW,Пирсон,нет,0.700,1.000,0.654,0.850,0.400
BoW,Жаккар,да,0.600,1.000,0.667,0.600,0.600
BoW,Косинус,да,0.667,1.000,0.667,0.800,0.400
BoW,Пирсон,да,0.667,1.000,0.667,0.800,0.400
TF-IDF,Жаккар,да,0.600,1.000,0.667,0.600,0.600
TF-IDF,Косинус,да,0.633,1.000,0.652,0.750,0.400
TF-IDF,Пирсон,да,0.633,1.000,0.652,0.750,0.400
TF-IDF,Спирмен,да,0.667,1.000,0.667,0.800,0.400


Метрики, выбранные автовыбором на test: {'Косинус': 16, 'Спирмен': 8, 'Жаккар': 5, 'Пирсон': 1}


Известные вопросы,Метрика,Отвечено верно,Всего
до 3 токенов,Жаккар,0,3
до 3 токенов,Косинус,1,3
до 3 токенов,Автовыбор,2,3
больше 3,Жаккар,10,17
больше 3,Косинус,9,17
больше 3,Автовыбор,12,17


In [6]:
def quality(stat):
    return (stat["Recall"] + stat["Отказы вне базы"]) / 2, stat["Accuracy"], stat["Precision"]

# Выбираем конфигурацию на validation, затем показываем результат на test.
recommended = max((key for key in configs if key[2]), key=lambda key: quality(stats_val[key]))
test_leaders = [key for key in configs if stats[key][0]["Accuracy"] == max(s[0]["Accuracy"] for s in stats.values())]
print("Выбор по validation:", recommended, "| test:", stats[recommended][0])
print("Максимальная Accuracy на test:", stats[test_leaders[0]][0]["Accuracy"])
print("Лидеры на test:", [(label, metric, cutoff) for label, metric, cutoff in test_leaders])

errors = []
for (question, want), got, hits in zip(test, stats[recommended][1], stats[recommended][2]):
    if got != want:
        errors.append((question, want or "нет", got or "отказ", hit_label(hits)))
if errors:
    table(["Ошибка выбранного подхода", "Эталон", "Ответ", "Первый кандидат до порога"], errors)

Выбор по validation: ('TF-IDF + парафразы', 'Пирсон', True) | test: {'Accuracy': 0.5666666666666667, 'Top-3 accuracy': 1.0, 'Precision': 0.6666666666666666, 'Recall': 0.5, 'Отказы вне базы': 0.7, 'Верных': 17}
Максимальная Accuracy на test: 0.7
Лидеры на test: [('BoW', 'Жаккар', False), ('BoW', 'Косинус', False), ('BoW', 'Пирсон', False)]


Ошибка выбранного подхода,Эталон,Ответ,Первый кандидат до порога
Как передать строку по chan между двумя задачами?,channel,отказ,channel (0.371)
Как не дать двум горутинам одновременно поменять общий счётчик?,mutex,отказ,mutex (0.457)
Как отменить запрос спустя пять секунд с помощью context?,context,отказ,context (0.405)
append среза,append,отказ,append (0.336)
Как создать map со строковыми ключами и целыми значениями?,map_init,map_check,map_check (0.536)
В каком порядке идут отложенные вызовы при выходе из функции?,defer,отказ,defer (0.470)
Реализация интерфейса без implements,interface,отказ,interface (0.471)
Как удалить лишние зависимости Go после удаления импортов?,mod_tidy,отказ,mod_tidy (0.446)
"Как поменять число по адресу, переданному в функцию?",pointer,отказ,channel (0.296)
Каким флагом ловить гонки при go test?,race,tests,tests (0.576)
